# 04 — Temporal Segmentation

This notebook walks through kernel change-point detection with PELT and the slope heuristic (Chapter 5, §5.3).

**Pipeline overview:**
1. **Config & Data Loading** — Load CPD experiment and deployment configurations
2. **Embedding Features** — Inspect Z-score normalized VideoMAE-v2 embeddings (D=1408, no PCA)
3. **Change-Point Detection** — KernelCPD with RBF kernel, PELT algorithm, penalty sweep
4. **Slope Heuristic** — Dataset-level penalty calibration via fixed-effects OLS (Arlot et al.)
5. **Deployment** — Apply slope heuristic, load final segmentation results
6. **Visualization** — Gram matrices, chronograms, cost curves
7. **Segment-Level Analysis** — Duration distributions, segment statistics

## 1. Config & Data Loading

Load change-point detection configurations (experiment sweep + deployment) and video embedding dataset.

See: `smartflat.configs.change_points_config`, `smartflat.engine.change_point_detection`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from pprint import pprint
from IPython.display import display
from scipy.spatial.distance import pdist, squareform
from sklearn.linear_model import LinearRegression

from smartflat.configs.loader import import_config
from smartflat.datasets.loader import get_dataset
from smartflat.engine.builders import build_model
from smartflat.engine.change_point_detection import (
    get_results_change_point_detection,
    main_deployment,
)
from smartflat.features.symbolization.utils import retrieve_segmentation_costs
from smartflat.features.symbolization.visualization import plot_segments_costs
from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils_visualization import plot_chronogames, plot_gram

In [ ]:
# --- Experiment parameters ---
annotator_id = 'samperochon'
round_number = 0

# Experiment config: penalty sweep with KernelCPD + RBF kernel (calibration mode)
experiment_config_name = 'KernelChangePointDetectionExperimentConfig'

# Deployment config: slope heuristic with dataset-level calibrated penalty
deployment_config_name = 'KernelChangePointDetectionDeploymentConfig'

# Data root
data_root = get_data_root()
print(f"Data root: {data_root}")

In [ ]:
# Load experiment config — penalty sweep used to estimate the slope heuristic
exp_config = import_config(experiment_config_name)
print(f"Experiment: {exp_config.experiment_name}")
print(f"Model: {exp_config.model_name} | Kernel: {exp_config.model_params['kernel']}")
print(f"Penalty mode: {exp_config.model_params['penalty']}")
print(f"\nModel params:")
pprint(exp_config.model_params)
print(f"\nDataset params:")
pprint(exp_config.dataset_params)

In [ ]:
# Load deployment config — uses slope heuristic with a pre-estimated global slope
deploy_config = import_config(deployment_config_name)
print(f"Deployment: {deploy_config.experiment_name}")
print(f"Penalty mode: {deploy_config.model_params['penalty']}")
print(f"Global slope: {deploy_config.model_params['global_slope_heuristics']:.6e}")
print(f"\nPenalty formula: lambda = -2 * slope * N")
print(f"  For N=1000: lambda = {-2 * deploy_config.model_params['global_slope_heuristics'] * 1000:.4f}")
print(f"  For N=5000: lambda = {-2 * deploy_config.model_params['global_slope_heuristics'] * 5000:.4f}")

## 2. Embedding Features

The final pipeline operates on Z-score normalized VideoMAE-v2 embeddings at full dimensionality
(D=1408). No PCA is applied — the kernel-based CPD handles the high-dimensional space directly.

See: `smartflat.datasets.dataset_video_representations.VideoBlockDataset`

In [ ]:
# Load the video block representation dataset
dset = get_dataset(
    dataset_name=exp_config.dataset_name,
    scenario=exp_config.dataset_params.get('scenario'),
    task_names=exp_config.dataset_params['task_names'],
    modality=exp_config.dataset_params['modality'],
    normalization=exp_config.dataset_params['normalization'],
    n_pca_components=exp_config.dataset_params.get('n_pca_components'),
    whiten=exp_config.dataset_params.get('whiten'),
)
print(f"Dataset: {len(dset)} recordings")
display(dset.metadata[['identifier', 'participant_id', 'task_name', 'modality',
                        'n_frames', 'fps', 'duration']].head(10))

In [ ]:
# Inspect a single sample's embedding and compute the RBF bandwidth (median heuristic)
if len(dset) > 0:
    X_sample, _, _ = dset[0]
    row_sample = dset.metadata.iloc[0]
    N_sample, D_sample = X_sample.shape

    # Median heuristic for RBF bandwidth
    temperature = exp_config.model_params.get('temperature', 1.0)
    condensed_sq_dists = pdist(X_sample, metric='sqeuclidean')
    sigma_rbf = np.median(np.sqrt(condensed_sq_dists))
    gamma_rbf = 1 / (2 * temperature * sigma_rbf ** 2)

    print(f"Sample: {row_sample.identifier}")
    print(f"Embedding shape: ({N_sample}, {D_sample}) — {N_sample} time steps, D={D_sample}")
    print(f"Duration: {row_sample.duration:.1f} min | FPS: {row_sample.fps}")
    print(f"Stats: mean={X_sample.mean():.3f}, std={X_sample.std():.3f}")
    print(f"\nRBF bandwidth (median heuristic):")
    print(f"  sigma = {sigma_rbf:.2f}, gamma = {gamma_rbf:.2e}, temperature = {temperature}")

## 3. Change-Point Detection (KCP / PELT)

The model uses `ruptures.KernelCPD` with an RBF kernel whose bandwidth is calibrated
via the median heuristic: γ = 1 / (2·τ·σ²) where σ = median pairwise Euclidean distance.

PELT provides exact, efficient change-point detection with a per-sample penalty.

See: `smartflat.engine.builders.build_model`, `smartflat.engine.change_point_detection`

In [ ]:
# Demonstrate CPD on a single sample with a fixed penalty
if len(dset) > 0:
    model_params_demo = {
        'kernel': 'rbf',
        'gamma': gamma_rbf,
        'min_size': exp_config.model_params['min_size'],
        'jump': exp_config.model_params['jump'],
    }
    model = build_model('KernelCPD', model_params_demo)

    # Detect change points at a fixed penalty
    penalty_demo = 10.0
    cpt = model.fit_predict(X_sample, pen=penalty_demo)[:-1]
    cpts_full = [0] + list(cpt) + [N_sample]

    print(f"Sample: {row_sample.identifier} (N={N_sample})")
    print(f"Penalty = {penalty_demo:.1f} → {len(cpt)} change points")
    print(f"First 10 boundaries: {cpts_full[:11]}{'...' if len(cpts_full) > 11 else ''}")

    # Segment durations in seconds
    seg_lengths = np.ediff1d(cpts_full)
    seg_durations = seg_lengths * row_sample.duration * 60 / N_sample
    print(f"Segment durations (s): median={np.median(seg_durations):.1f}, "
          f"min={seg_durations.min():.1f}, max={seg_durations.max():.1f}")

In [ ]:
# Penalty sweep on a single sample — penalty-complexity curve
if len(dset) > 0:
    penalty_list = np.logspace(-1, 2, 40)
    n_cpts_list = []

    for pen in penalty_list:
        mdl = build_model('KernelCPD', model_params_demo)
        cpts_pen = mdl.fit_predict(X_sample, pen=pen)[:-1]
        n_cpts_list.append(len(cpts_pen))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(penalty_list, n_cpts_list, 'o-', markersize=3)
    ax.set_xscale('log')
    ax.set(xlabel='Penalty (log scale)', ylabel='Number of change points',
           title=f'Penalty-complexity curve — {row_sample.identifier}')
    plt.tight_layout()
    plt.show()

## 4. Slope Heuristic Estimation

The slope heuristic (Arlot et al., 2019) selects the optimal penalty by estimating the
slope of the empirical risk R_N as a function of model complexity D_m (number of segments).

**Method:**
1. Run CPD at many penalty values → get (penalty, n_cpts, fit_cost) per participant
2. Regress R_N/N ~ D_m using fixed-effects OLS (one intercept per participant, shared slope)
3. Restrict regression to D ∈ [0.6·D_max, D_max] (linear regime)
4. Optimal penalty: λ̂_opt = −2 · β̂ · N (where β̂ is the estimated slope)

This provides a **single dataset-level penalty** that scales with sequence length N,
ensuring fair comparison across participants with varying video durations.

See: `smartflat.engine.change_point_detection.get_results_change_point_detection`,
     `smartflat.features.symbolization.utils.retrieve_segmentation_costs`

In [ ]:
# Load pre-computed experiment results (penalty sweep across all participants)
results = get_results_change_point_detection(
    exp_config,
    annotator_id=annotator_id,
    round_number=round_number,
    use_stored=True,
)

if results is not None and len(results) > 0:
    print(f"Results: {results.shape[0]} rows, {results.identifier.nunique()} participants, "
          f"{results.penalty.nunique()} penalty values")
    display(results[['identifier', 'task_name', 'modality', 'penalty', 'n_cpts',
                      'N', 'duration']].head(10))
else:
    print("No stored experiment results found. Run the penalty sweep first (HPC).")
    results = None

In [ ]:
# Compute segmentation costs (fit_cost, reg_cost, total_cost) from the penalty sweep
if results is not None:
    results_costs = retrieve_segmentation_costs(
        results,
        'K_space',  # input_space (placeholder for raw embedding space)
        config_name=experiment_config_name,
        temporal_segmentation_col='cpts',
        penalty_column='penalty',
        annotator_id=annotator_id,
        round_number=round_number,
        include_segments=False,
        suffix='',
    )
    cost_cols = [c for c in results_costs.columns if 'cost' in c.lower()]
    print(f"Cost columns added: {cost_cols}")
    display(results_costs[['identifier', 'penalty', 'n_cpts', 'N'] + cost_cols].head(10))
else:
    results_costs = None

In [ ]:
# --- Slope heuristic estimation functions ---
# Adapted from thesis notebook (loss-optimization/analysis_change_point_space.ipynb)


def prepare_regression_data(Rn, D_m, n, group=None, Dmax=None, verbose=False):
    """Prepare pooled regression data, masking to D ∈ [0.6·Dmax, Dmax] per group."""
    Rn = np.asarray(Rn) / np.asarray(n)
    D_m = np.asarray(D_m)
    group = None if group is None else np.asarray(group)

    keep_mask = np.zeros(len(Rn), dtype=bool)

    if group is None:
        _Dmax = Dmax if Dmax is not None else D_m.max()
        mask = (D_m >= 0.6 * _Dmax) & (D_m <= _Dmax)
        keep_mask |= mask
    else:
        for g in np.unique(group):
            idx = (group == g)
            Dmax_g = Dmax if Dmax is not None else D_m[idx].max()
            Dmax_g = min(Dmax_g, D_m[idx].max())
            mask_g = (D_m[idx] >= 0.6 * Dmax_g) & (D_m[idx] <= Dmax_g)
            keep_mask[idx] = mask_g
            if verbose:
                print(f"Group {g}: kept {mask_g.sum()} / {idx.sum()} points (Dmax={Dmax_g})")

    if keep_mask.sum() < 3:
        raise ValueError("Too few points in regression window.")

    return Rn[keep_mask], D_m[keep_mask], (None if group is None else group[keep_mask])


def estimate_slope_heuristic_group(Rn, D_m, n, group=None, verbose=False, plot=False,
                                   title=None, alpha=2.0):
    """
    Estimate the slope heuristic via fixed-effects OLS regression.

    Regresses (Rn/n) ~ D_m with per-group intercepts and a shared slope,
    restricted to D ∈ [0.6·Dmax, Dmax].

    Returns: (slope_opt, intercepts) where slope_opt = alpha * slope.
    """
    Rn, D_m, group = prepare_regression_data(Rn, D_m, n, group=group, verbose=verbose)
    Dmax = D_m.max()

    if group is None:
        raise ValueError("group must be provided for fixed-effects estimation.")

    # Build design matrix: [group_dummies | D]
    df_reg = pd.DataFrame({'Y': Rn, 'D': D_m, 'group': group})
    X_grp = pd.get_dummies(df_reg['group'], drop_first=False).astype(float)
    X = X_grp.copy()
    X['D'] = df_reg['D'].astype(float)

    # Fit OLS (no explicit constant — group dummies absorb it)
    ols_model = sm.OLS(df_reg['Y'].values, X.values).fit()
    params = pd.Series(ols_model.params, index=X.columns)

    slope = float(params['D'])
    slope_opt = alpha * slope
    intercepts = {col: float(params[col]) for col in X_grp.columns}
    R2 = ols_model.rsquared

    if plot:
        plt.figure(figsize=(12, 5))
        plt.scatter(D_m, Rn, alpha=0.3, s=5, label="all data")
        D_line = np.linspace(D_m.min(), D_m.max(), 80)
        for g in list(pd.unique(group))[:5]:
            plt.plot(D_line, intercepts[g] + slope * D_line, lw=1)
        plt.axvline(0.6 * Dmax, color='grey', ls='--', lw=1, label='fit interval')
        plt.axvline(Dmax, color='grey', ls='--', lw=1)
        plt.xlabel('D (number of segments)')
        plt.ylabel('R_N / N')
        plt.title(f'Slope × {alpha} = {slope_opt:.6e} | R² = {R2:.4f}'
                  if title is None else f'{title} | Slope × {alpha} = {slope_opt:.6e}')
        plt.legend()
        plt.show()

    print(f"Slope (s) = {slope:.6g}, slope × {alpha} = {slope_opt:.6g}, R² = {R2:.4f}")
    return slope_opt, intercepts


def estimate_slope_heuristic_individual(Rn, D_m, n, verbose=False, plot=False, title=None):
    """Estimate slope by regressing (Rn/n) ~ D_m over D ∈ [0.6·Dmax, Dmax] for one participant."""
    Rn = np.array(Rn) / np.array(n)
    D_m = np.array(D_m)
    Dmax = D_m.max()
    mask = (D_m >= 0.6 * Dmax) & (D_m <= Dmax)

    reg = LinearRegression().fit(D_m[mask].reshape(-1, 1), Rn[mask])
    slope = reg.coef_[0]
    R2 = reg.score(D_m[mask].reshape(-1, 1), Rn[mask])

    if verbose:
        print(f"Slope (s) = {slope:.4f}, R² = {R2:.4f}")

    if plot:
        plt.scatter(D_m, Rn, alpha=0.6, s=10)
        plt.plot(D_m, reg.predict(D_m.reshape(-1, 1)), color='red', lw=2, label=f'R²={R2:.3f}')
        plt.axvline(0.6 * Dmax, color='grey', ls='--', lw=1)
        plt.axvline(Dmax, color='grey', ls='--', lw=1)
        plt.xlabel('D (number of segments)')
        plt.ylabel('R_N / N')
        plt.title(f'Slope={slope:.4f}' if title is None else f'{title} | Slope={slope:.4f}')
        plt.legend()
        plt.show()

    return slope

In [ ]:
# Estimate the global slope using fixed-effects OLS across all participants
if results_costs is not None:
    slope_opt, intercepts = estimate_slope_heuristic_group(
        Rn=results_costs.fit_cost,
        D_m=results_costs.n_cpts,
        n=results_costs.N,
        group=results_costs.participant_id,
        verbose=False,
        plot=True,
        alpha=2.0,
    )
    print(f"\nEstimated global slope: {slope_opt:.6e}")
    print(f"Config value:           {deploy_config.model_params['global_slope_heuristics']:.6e}")
else:
    slope_opt = None

In [ ]:
# Show how the slope translates to per-participant penalties
if results_costs is not None and slope_opt is not None:
    slope_config = deploy_config.model_params['global_slope_heuristics']
    unique_participants = results_costs.drop_duplicates('identifier')

    unique_participants = unique_participants.copy()
    unique_participants['penalty_estimated'] = -2 * slope_opt * unique_participants['N']
    unique_participants['penalty_config'] = -2 * slope_config * unique_participants['N']

    print("Per-participant penalties (first 10):")
    display(unique_participants[['identifier', 'N', 'duration',
                                  'penalty_estimated', 'penalty_config']].head(10))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(unique_participants['N'], unique_participants['penalty_config'], s=15, alpha=0.7)
    ax.set(xlabel='Sequence length N', ylabel='Penalty λ = −2·slope·N',
           title='Penalty scales linearly with sequence length')
    plt.tight_layout()
    plt.show()

## 5. Deployment Results

Load the final segmentation produced by the slope heuristic deployment.

See: `smartflat.engine.change_point_detection.get_results_change_point_detection`

In [ ]:
# Load deployment results (segmentation with slope heuristic penalty)
results_deploy = get_results_change_point_detection(
    deploy_config,
    annotator_id=annotator_id,
    round_number=round_number,
    use_stored=True,
)

if results_deploy is not None and len(results_deploy) > 0:
    print(f"Deployment results: {results_deploy.shape[0]} rows, "
          f"{results_deploy.identifier.nunique()} participants")
    display(results_deploy[['identifier', 'task_name', 'modality', 'penalty',
                             'n_cpts', 'N', 'duration']].head(10))
else:
    print("No deployment results found. Run main_deployment() first.")
    results_deploy = None

In [ ]:
# Summary statistics per task/modality
if results_deploy is not None:
    summary = (
        results_deploy
        .groupby(['task_name', 'modality'])
        .agg(
            n_participants=('identifier', 'nunique'),
            mean_n_cpts=('n_cpts', 'mean'),
            std_n_cpts=('n_cpts', 'std'),
            mean_duration_min=('duration', 'mean'),
            mean_cpts_freq=('cpts_frequency', 'mean'),
        )
        .round(2)
    )
    print("Segmentation summary:")
    display(summary)

In [ ]:
# Segment duration statistics
if results_deploy is not None:
    results_deploy = results_deploy.copy()
    results_deploy['segments_length'] = results_deploy['cpts'].apply(np.ediff1d)
    # Convert to seconds: segment_length_in_steps * (duration_min * 60 / N)
    results_deploy['segments_duration_s'] = results_deploy.apply(
        lambda r: r['segments_length'] * r['duration'] * 60 / r['N'], axis=1)

    all_durations = np.hstack(results_deploy['segments_duration_s'].values)
    print(f"Total segments: {len(all_durations)}")
    print(f"Duration quantiles (seconds):")
    quantiles = [0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    for q in quantiles:
        print(f"  {q:.0%}: {np.quantile(all_durations, q):.1f}s")

## 6. Visualization

Plot Gram matrices with segment boundaries, per-segment cost bars, and temporal chronograms.

See: `smartflat.utils.utils_visualization.plot_gram`, `smartflat.utils.utils_visualization.plot_chronogames`,
     `smartflat.features.symbolization.visualization.plot_segments_costs`

In [ ]:
# Gram matrix (embedding self-similarity) with change-point boundaries
if results_deploy is not None:
    row_viz = results_deploy.iloc[0]
    plot_gram(
        row_viz,
        temporal_segmentation_col='cpts',
        title=f'{row_viz.identifier} — Gram matrix with CPD segments',
    )
    plt.show()

In [ ]:
# Per-segment cost bars for one participant
if results_costs is not None:
    # Use the deployment result with costs (pick one participant at the best penalty)
    row_cost = results_costs.groupby('identifier').first().iloc[0]
    plot_segments_costs(row_cost, n_start=0, n_end=None)
    plt.suptitle(f'Per-segment costs — {row_cost.name}', y=1.02)
    plt.show()

In [ ]:
# Chronograms — temporal segmentation for several participants
if results_deploy is not None:
    plot_chronogames(results_deploy, labels_col='cpts', n_subjects=6)
    plt.suptitle('Temporal segmentation (slope heuristic deployment)', y=1.02)
    plt.show()

In [ ]:
# Cost decomposition: fit_cost vs n_cpts across the penalty sweep
if results_costs is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    participants = results_costs.participant_id.unique()
    colors = plt.cm.tab20(np.linspace(0, 1, min(20, len(participants))))

    for i, pid in enumerate(participants[:20]):
        group = results_costs[results_costs.participant_id == pid].sort_values('n_cpts')
        ax.scatter(group['n_cpts'], group['fit_cost'] / group['N'],
                   s=5, alpha=0.4, color=colors[i % len(colors)])

    ax.set(xlabel='D (number of segments)', ylabel='R_N / N (normalized fit cost)',
           title='Cost decomposition across penalty sweep')
    plt.tight_layout()
    plt.show()

In [ ]:
# Individual vs global slope comparison for a few participants
if results_costs is not None:
    sample_pids = results_costs.participant_id.unique()[:4]
    fig, axes = plt.subplots(1, len(sample_pids), figsize=(5 * len(sample_pids), 4))
    if len(sample_pids) == 1:
        axes = [axes]

    for ax, pid in zip(axes, sample_pids):
        group = results_costs[results_costs.participant_id == pid]
        plt.sca(ax)
        slope_indiv = estimate_slope_heuristic_individual(
            group.fit_cost, group.n_cpts, group.N,
            verbose=True, plot=True, title=pid,
        )

    plt.tight_layout()
    plt.show()

## 7. Segment-Level Analysis

Analyze the distribution of segment durations and the relationship between
number of segments and video duration.

See: `smartflat.engine.change_point_detection`

In [ ]:
# Segment duration distribution
if results_deploy is not None:
    all_durations = np.hstack(results_deploy['segments_duration_s'].values)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Histogram (log scale x-axis)
    axes[0].hist(all_durations[all_durations > 0], bins=100, edgecolor='black', alpha=0.7)
    axes[0].set_xscale('log')
    axes[0].set(xlabel='Segment duration (s, log scale)', ylabel='Count',
                title=f'Segment duration distribution (N={len(all_durations)})')
    axes[0].axvline(np.median(all_durations), color='red', linestyle='--',
                    label=f'Median = {np.median(all_durations):.1f}s')
    axes[0].legend()

    # ECDF
    sorted_dur = np.sort(all_durations)
    ecdf = np.arange(1, len(sorted_dur) + 1) / len(sorted_dur)
    axes[1].plot(sorted_dur, ecdf, lw=1.5)
    axes[1].set_xscale('log')
    axes[1].set(xlabel='Segment duration (s, log scale)', ylabel='Cumulative probability',
                title='Empirical CDF of segment durations')
    axes[1].axhline(0.5, color='grey', linestyle=':', alpha=0.5)

    plt.tight_layout()
    plt.show()

In [ ]:
# Number of segments vs video duration — should be approximately linear (slope heuristic)
if results_deploy is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(results_deploy['duration'], results_deploy['n_cpts'], s=15, alpha=0.6)

    # Linear fit
    x = results_deploy['duration'].values
    y = results_deploy['n_cpts'].values
    mask = np.isfinite(x) & np.isfinite(y)
    reg = LinearRegression().fit(x[mask].reshape(-1, 1), y[mask])
    x_line = np.linspace(x[mask].min(), x[mask].max(), 100)
    ax.plot(x_line, reg.predict(x_line.reshape(-1, 1)), 'r--', lw=2,
            label=f'Linear fit: {reg.coef_[0]:.1f} segments/min + {reg.intercept_:.1f}')

    ax.set(xlabel='Video duration (min)', ylabel='Number of segments',
           title='Segments vs duration (slope heuristic ensures proportionality)')
    ax.legend()
    plt.tight_layout()
    plt.show()